# LGBM Optuna + SHAP + 원본 비교

이 노트북의 목적은 다음 3가지다.

1. 원본 LGBM 결과와 같은 기준으로 새 LGBM을 다시 튜닝한다.
2. 튜닝된 LGBM으로 SHAP 분석을 수행한다.
3. 원본 결과와 새 결과를 비교표로 정리한다.

중요: 원본과 공정하게 비교하려면 `FEATURE_COLS_36`에 원본 36개 feature 이름을 직접 넣는 것이 가장 좋다.

In [1]:
# 필요 시 최초 1회만 실행
# import sys
# !{sys.executable} -m pip install lightgbm optuna shap pandas numpy scikit-learn matplotlib

In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import optuna
import shap

In [3]:
RANDOM_STATE = 42
TARGET_COL = "oil_diff_target"
DATE_COL = "date"

N_SPLITS = 5
N_TRIALS = 40

ORIGINAL_RESULTS = pd.DataFrame([
    {
        "version": "original_default_36feat",
        "dataset": "dataset4_36_features",
        "model": "LGBM_default",
        "feature_count": 36,
        "R2_mean": -0.0044,
        "RMSE_mean": 1.8014,
        "MAE_mean": 1.1529,
        "note": "기존 원본 baseline"
    },
    {
        "version": "original_tuned_36feat",
        "dataset": "dataset4_36_features",
        "model": "LGBM_optuna",
        "feature_count": 36,
        "R2_mean": 0.0001,
        "RMSE_mean": 1.7976,
        "MAE_mean": 1.1515,
        "note": "기존 원본 Optuna 40 trials"
    },
])

# 원본 36개 변수명이 있으면 여기에 직접 넣기
# 예: FEATURE_COLS_36 = ["OilInventories", "VIX", "OilPrice", ...]
FEATURE_COLS_36 = None

In [4]:
def find_data_dir():
    candidates = [
        Path("../data/Finance_Final"),
        Path("data/Finance_Final"),
        Path("./data/Finance_Final"),
        Path("../../data/Finance_Final"),
        Path.cwd() / "../data/Finance_Final",
        Path.cwd() / "data/Finance_Final",
    ]

    for p in candidates:
        if p.exists():
            return p.resolve()

    raise FileNotFoundError(
        "Finance_Final 데이터 폴더를 찾지 못했습니다. "
        f"현재 작업 폴더: {Path.cwd()}"
    )


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("../outputs/lgbm_optuna_shap_compare")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("사용 중인 DATA_DIR:", DATA_DIR)
print("결과 저장 OUTPUT_DIR:", OUTPUT_DIR.resolve())

사용 중인 DATA_DIR: C:\Users\user\Desktop\비어플\baf_project\finance2\project\data\Finance_Final
결과 저장 OUTPUT_DIR: C:\Users\user\Desktop\비어플\baf_project\finance2\project\outputs\lgbm_optuna_shap_compare


In [5]:
def load_dataset4():
    candidates = [
        "dataset4_derived_full.csv",
        "dataset4_derived_full_with_dummies.csv",
    ]

    for fname in candidates:
        path = DATA_DIR / fname
        if path.exists():
            df = pd.read_csv(path)
            print(f"불러온 파일: {path}")
            return df, fname

    raise FileNotFoundError(
        "dataset4 파일을 찾지 못했습니다. "
        "dataset4_derived_full.csv 또는 dataset4_derived_full_with_dummies.csv가 있는지 확인하세요."
    )


def prepare_dataset(df, feature_cols=None):
    df = df.copy()

    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
        df = df.sort_values(DATE_COL).reset_index(drop=True)

    if TARGET_COL not in df.columns:
        raise ValueError(f"{TARGET_COL} 컬럼이 없습니다.")

    if feature_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

        drop_keywords = ["target", "future"]
        feature_cols = []

        for c in numeric_cols:
            if c == TARGET_COL:
                continue

            lower_c = c.lower()
            if any(k in lower_c for k in drop_keywords):
                continue

            feature_cols.append(c)

        print("자동 선택 feature 수:", len(feature_cols))
        print("공정 비교를 원하면 FEATURE_COLS_36에 원본 36개 변수명을 직접 넣는 것을 권장.")
    else:
        missing = [c for c in feature_cols if c not in df.columns]
        if missing:
            raise ValueError(f"데이터에 없는 feature가 있습니다: {missing}")

    keep_cols = []
    if DATE_COL in df.columns:
        keep_cols.append(DATE_COL)
    keep_cols += [TARGET_COL] + feature_cols

    model_df = df[keep_cols].dropna().reset_index(drop=True)

    X = model_df[feature_cols].copy()
    y = model_df[TARGET_COL].copy()

    return model_df, X, y, feature_cols

In [6]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def directional_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    true_sign = np.sign(y_true)
    pred_sign = np.sign(y_pred)

    mask = (true_sign != 0) & (pred_sign != 0)
    if mask.sum() == 0:
        return np.nan

    return float(np.mean(true_sign[mask] == pred_sign[mask]))


def metric_row(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "Directional_Accuracy": directional_accuracy(y_true, y_pred),
    }


def make_lgbm(params=None):
    base_params = {
        "objective": "regression",
        "n_estimators": 4000,
        "learning_rate": 0.01,
        "num_leaves": 15,
        "max_depth": 4,
        "min_child_samples": 80,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_split_gain": 0.0,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
        "force_col_wise": True,
    }

    if params is not None:
        base_params.update(params)

    return LGBMRegressor(**base_params)


def oos_cv_evaluate(X, y, params=None, model_label="LGBM"):
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)

    rows = []
    oos_pred = pd.Series(index=y.index, dtype=float)
    best_iters = []

    for fold, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = make_lgbm(params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="rmse",
            callbacks=[
                early_stopping(stopping_rounds=100, verbose=False),
                log_evaluation(period=0),
            ],
        )

        pred = model.predict(X_valid)
        oos_pred.iloc[valid_idx] = pred

        row = metric_row(y_valid, pred)
        row.update({
            "fold": fold,
            "model": model_label,
            "best_iteration": getattr(model, "best_iteration_", None),
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
        })
        rows.append(row)
        best_iters.append(getattr(model, "best_iteration_", None))

    fold_result = pd.DataFrame(rows)

    summary = {
        "model": model_label,
        "R2_mean": fold_result["R2"].mean(),
        "RMSE_mean": fold_result["RMSE"].mean(),
        "MAE_mean": fold_result["MAE"].mean(),
        "Directional_Accuracy_mean": fold_result["Directional_Accuracy"].mean(),
        "best_iter_list": "/".join(map(str, best_iters)),
    }

    return pd.DataFrame([summary]), fold_result, oos_pred

In [7]:
def tune_lgbm_optuna(X, y, n_trials=N_TRIALS):
    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 3, 7)
        max_num_leaves = min(2 ** max_depth - 1, 80)

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, max_num_leaves),
            "max_depth": max_depth,
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 250),
            "subsample": trial.suggest_float("subsample", 0.55, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 50.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.3),
        }

        summary, _, _ = oos_cv_evaluate(X, y, params=params, model_label="trial")
        return summary.loc[0, "RMSE_mean"]

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    return study.best_params, study.best_value, study

In [8]:
def fit_final_model_for_shap(X, y, best_params):
    model = make_lgbm(best_params)
    model.fit(X, y)
    return model


def run_shap_analysis(model, X, feature_cols, max_display=20):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    shap_df = pd.DataFrame(shap_values, columns=feature_cols)

    shap_importance = (
        shap_df.abs()
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    shap_importance.columns = ["feature", "mean_abs_SHAP"]
    shap_importance["rank"] = np.arange(1, len(shap_importance) + 1)

    shap_importance.to_csv(OUTPUT_DIR / "shap_importance.csv", index=False)

    plt.figure()
    shap.summary_plot(shap_values, X, show=False, max_display=max_display)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_summary_plot.png", dpi=200, bbox_inches="tight")
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values, X, plot_type="bar", show=False, max_display=max_display)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_bar_plot.png", dpi=200, bbox_inches="tight")
    plt.close()

    return shap_df, shap_importance

In [9]:
df, dataset_file = load_dataset4()
model_df, X, y, feature_cols = prepare_dataset(df, FEATURE_COLS_36)

print("분석 데이터 shape:", model_df.shape)
print("feature count:", len(feature_cols))

불러온 파일: C:\Users\user\Desktop\비어플\baf_project\finance2\project\data\Finance_Final\dataset4_derived_full.csv
자동 선택 feature 수: 26
공정 비교를 원하면 FEATURE_COLS_36에 원본 36개 변수명을 직접 넣는 것을 권장.
분석 데이터 shape: (4798, 28)
feature count: 26


In [10]:
# 1) 새 default LGBM OOS 평가
default_summary, default_fold, default_oos_pred = oos_cv_evaluate(
    X, y,
    params=None,
    model_label="new_default"
)

default_fold.to_csv(OUTPUT_DIR / "new_default_fold_result.csv", index=False)
display(default_summary)
display(default_fold)

,model,R2_mean,RMSE_mean,MAE_mean,Directional_Accuracy_mean,best_iter_list
0,new_default,-0.000214,1.79783,1.151928,0.504768,10/50/9/14/1


,R2,RMSE,MAE,Directional_Accuracy,fold,model,best_iteration,n_train,n_valid
0,0.000649,1.621983,1.198840,0.524467,1,new_default,10,803,799
1,-0.002221,1.335897,1.024462,0.463568,2,new_default,50,1602,799
2,0.000593,1.118767,0.805750,0.499368,3,new_default,9,2401,799
3,0.000162,3.303617,1.553859,0.519497,4,new_default,14,3200,799
4,-0.000252,1.608889,1.176731,0.516939,5,new_default,1,3999,799


In [11]:
# 2) Optuna 튜닝
best_params, best_cv_rmse, study = tune_lgbm_optuna(X, y, n_trials=N_TRIALS)

best_params_df = pd.DataFrame([{
    "best_cv_rmse": best_cv_rmse,
    **best_params
}])
best_params_df.to_csv(OUTPUT_DIR / "new_optuna_best_params.csv", index=False)

display(best_params_df)

[I 2026-05-16 02:27:26,285] A new study created in memory with name: no-name-08465da0-b03f-453e-a002-34065d742702


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-05-16 02:27:26,725] Trial 0 finished with value: 1.7934207958601074 and parameters: {'max_depth': 4, 'learning_rate': 0.044635901521768134, 'num_leaves': 13, 'min_child_samples': 154, 'subsample': 0.6202083881990965, 'subsample_freq': 2, 'colsample_bytree': 0.5761376254756898, 'reg_alpha': 1.156732719914599, 'reg_lambda': 0.04245867573059574, 'min_split_gain': 0.21242177333881365}. Best is trial 0 with value: 1.7934207958601074.
[I 2026-05-16 02:27:26,969] Trial 1 finished with value: 1.7967313354974894 and parameters: {'max_depth': 3, 'learning_rate': 0.04665303012212833, 'num_leaves': 7, 'min_child_samples': 210, 'subsample': 0.6455525998052243, 'subsample_freq': 2, 'colsample_bytree': 0.6325320294340453, 'reg_alpha': 0.000134801802908908, 'reg_lambda': 0.010966903618110112, 'min_split_gain': 0.12958350559263473}. Best is trial 0 with value: 1.7934207958601074.
[I 2026-05-16 02:27:27,260] Trial 2 finished with value: 1.7954430110907544 and parameters: {'max_depth': 4, 'learni

,best_cv_rmse,max_depth,learning_rate,num_leaves,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,1.793421,4,0.044636,13,154,0.620208,2,0.576138,1.156733,0.042459,0.212422


In [12]:
# 3) tuned LGBM OOS 평가
tuned_summary, tuned_fold, tuned_oos_pred = oos_cv_evaluate(
    X, y,
    params=best_params,
    model_label="new_optuna"
)

tuned_fold.to_csv(OUTPUT_DIR / "new_optuna_fold_result.csv", index=False)
display(tuned_summary)
display(tuned_fold)

,model,R2_mean,RMSE_mean,MAE_mean,Directional_Accuracy_mean,best_iter_list
0,new_optuna,0.003043,1.793421,1.155979,0.505542,1/4/25/582/1


,R2,RMSE,MAE,Directional_Accuracy,fold,model,best_iteration,n_train,n_valid
0,0.000121,1.622411,1.198611,0.526976,1,new_optuna,1,803,799
1,-0.003939,1.337042,1.024439,0.459799,2,new_optuna,4,1602,799
2,0.007357,1.114974,0.806198,0.512010,3,new_optuna,25,2401,799
3,0.012314,3.283479,1.573750,0.527044,4,new_optuna,582,3200,799
4,-0.000636,1.609198,1.176898,0.501882,5,new_optuna,1,3999,799


In [13]:
# 4) 원본 vs 새 실행 비교표
new_compare = pd.concat([default_summary, tuned_summary], ignore_index=True)
new_compare["version"] = new_compare["model"]
new_compare["dataset"] = dataset_file
new_compare["feature_count"] = len(feature_cols)
new_compare["note"] = "이번 노트북 재실행 결과"

compare_cols = [
    "version", "dataset", "model", "feature_count",
    "R2_mean", "RMSE_mean", "MAE_mean", "Directional_Accuracy_mean",
    "best_iter_list", "note"
]

original = ORIGINAL_RESULTS.copy()
for c in compare_cols:
    if c not in original.columns:
        original[c] = np.nan

final_compare = pd.concat([
    original[compare_cols],
    new_compare[compare_cols]
], ignore_index=True)

final_compare.to_csv(OUTPUT_DIR / "original_vs_new_lgbm_compare.csv", index=False)
display(final_compare)

,version,dataset,model,feature_count,R2_mean,RMSE_mean,MAE_mean,Directional_Accuracy_mean,best_iter_list,note
0,original_default_36feat,dataset4_36_features,LGBM_default,36,-0.004400,1.801400,1.152900,NaN,NaN,기존 원본 baseline
1,original_tuned_36feat,dataset4_36_features,LGBM_optuna,36,0.000100,1.797600,1.151500,NaN,NaN,기존 원본 Optuna 40 trials
2,new_default,dataset4_derived_full.csv,new_default,26,-0.000214,1.797830,1.151928,0.504768,10/50/9/14/1,이번 노트북 재실행 결과
3,new_optuna,dataset4_derived_full.csv,new_optuna,26,0.003043,1.793421,1.155979,0.505542,1/4/25/582/1,이번 노트북 재실행 결과


In [14]:
# 5) SHAP 분석
final_model = fit_final_model_for_shap(X, y, best_params)
shap_df, shap_importance = run_shap_analysis(final_model, X, feature_cols, max_display=20)

display(shap_importance.head(20))

,feature,mean_abs_SHAP,rank
0,CPI,0.210080,1
1,MA_5,0.169824,2
2,oil_volatility_5,0.157295,3
3,MA_ratio,0.151033,4
4,OilPrice,0.149068,5
5,oil_momentum_20,0.137033,6
6,CPE,0.135398,7
7,OilInventories,0.133364,8
8,oil_diff_lag5,0.129964,9
9,oil_diff_lag1,0.122055,10


## 해석 기준

- `new_optuna`가 원본 tuned보다 RMSE가 낮으면 새 튜닝이 개선된 것.
- RMSE는 낮아졌지만 R²가 0 근처면 예측력 개선은 제한적이라고 해석.
- SHAP TOP 변수와 OLS 유의 변수가 겹치면 robust한 변수.
- OLS에는 없는데 SHAP 상위면 비선형/상호작용 가능성이 있는 변수.